# Temporal Client Exploration Playground
This notebook demonstrates direct interaction with the Temporal orchestrator using the `temporalio` client library.

It covers:
1. Connecting to the Temporal server
2. Listing active & historical workflow executions
3. Programmatically triggering a workflow
4. Querying a specific workflow's execution history

> **Note:** Jupyter already runs its own asyncio event loop, so we use `await` directly at the top level instead of `asyncio.run()`. This is supported natively in Jupyter kernels.

In [1]:
import sys
from pathlib import Path

# Locate and append project root so src.ragforge is importable
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

project_root = get_project_root().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/sunnyraj/code_files/git_repos/RagForge


In [2]:
import uuid
from temporalio.client import Client
from src.ragforge.config import TEMPORAL_URL

print(f"Temporal URL: {TEMPORAL_URL}")

Temporal URL: localhost:7233


## 1. Connect to Temporal Server

In [3]:
# Jupyter supports top-level `await` natively — no asyncio.run() needed
client = await Client.connect(TEMPORAL_URL)
print(f"Connected to Temporal server at: {TEMPORAL_URL}")

Connected to Temporal server at: localhost:7233


## 2. List Recent Workflow Executions

In [4]:
print("Listing recent Temporal workflow executions...\n")

count = 0
async for workflow in client.list_workflows():
    print(
        f"ID: {workflow.id}\n"
        f"  RunID  : {workflow.run_id}\n"
        f"  Type   : {workflow.workflow_type}\n"
        f"  Status : {workflow.status}\n"
        f"  Start  : {workflow.start_time}\n"
    )
    count += 1
    if count >= 10:  # Limit output to 10 workflows
        print("(showing first 10 results)")
        break

if count == 0:
    print("No workflow executions found. Start a workflow first!")

Listing recent Temporal workflow executions...

ID: test-op-create-27e98e9e-d4c4-405f-80bb-f6c06fb8935c
  RunID  : 019ede55-9563-74fe-921f-dbda52443c85
  Type   : OpenProjectWriteWorkflow
  Status : 1
  Start  : 2026-06-19 05:23:31.299329+00:00

ID: test-ingest-9c69d382-7b7a-481c-96af-aee207203c85
  RunID  : 019ede54-bf05-762f-b8f3-66f343028212
  Type   : IngestionWorkflow
  Status : 1
  Start  : 2026-06-19 05:22:36.421405+00:00

ID: test-ingest-f8520aeb-dc62-4718-be4d-935c0d712357
  RunID  : 019ede54-0510-72f2-93ad-d17f148ad4b8
  Type   : IngestionWorkflow
  Status : 1
  Start  : 2026-06-19 05:21:48.816193+00:00

ID: test-ingest-9d7744a0-2233-4ac9-9767-3f9283369f48
  RunID  : 019ede52-f072-7e3b-89b9-7d3599100913
  Type   : IngestionWorkflow
  Status : 1
  Start  : 2026-06-19 05:20:38.002933+00:00

ID: test-op-create-b5ae7bd4-a7e0-4d5e-94e2-fae337d5a36a
  RunID  : 019ede51-d058-72cc-b407-4dba417c1a75
  Type   : OpenProjectWriteWorkflow
  Status : 1
  Start  : 2026-06-19 05:19:24.248184

## 3. Describe a Specific Workflow by ID

Set `workflow_id` below to inspect a specific execution's history.

In [5]:
# Replace with an actual workflow ID from the listing above
workflow_id_to_inspect = None  # e.g. "ingest-documents-abc123"

if workflow_id_to_inspect:
    handle = client.get_workflow_handle(workflow_id_to_inspect)
    desc = await handle.describe()
    print(f"Workflow ID  : {desc.id}")
    print(f"Run ID       : {desc.run_id}")
    print(f"Status       : {desc.status}")
    print(f"Type         : {desc.workflow_type}")
    print(f"Task Queue   : {desc.task_queue}")
    print(f"Start Time   : {desc.start_time}")
    print(f"Close Time   : {desc.close_time}")
else:
    print("Set workflow_id_to_inspect to a real workflow ID from the listing above.")

Set workflow_id_to_inspect to a real workflow ID from the listing above.


## 4. Trigger an OpenProject Write Workflow

> **Requires**: Temporal Worker (`src/ragforge/worker.py`) must be running.

In [8]:
workflow_id = f"notebook-test-op-task-{uuid.uuid4()}"

handle = await client.start_workflow(
    "OpenProjectWriteWorkflow",
    arg={
        "action": "create",
        "project_id": "1",
        "title": "Jupyter Workflow Test Task",
        "description": "Task triggered programmatically via Jupyter notebook playground.",
        "task_type": "Task"
    },
    id=workflow_id,
    task_queue="ragforge-tasks"
)

print(f"Workflow started successfully!")
print(f"  Workflow ID    : {handle.id}")
print(f"  First Run ID   : {handle.first_execution_run_id}")
print(f"  Track it at    : http://localhost:8080")

Workflow started successfully!
  Workflow ID    : notebook-test-op-task-05f7f29a-bee6-4a87-8410-0add5e382559
  First Run ID   : 019ededa-4588-70ae-a42c-bdfddfe04f43
  Track it at    : http://localhost:8080


## 5. Wait for Workflow Result (Optional)

In [7]:
# This will block until the workflow completes (or times out)
# Only run this if you started a workflow in cell 4 above

try:
    result = await handle.result()
    print(f"Workflow completed! Result: {result}")
except Exception as e:
    print(f"Workflow failed or not yet started: {e}")

Workflow completed! Result: {'id': 38, 'status': 'New', 'subject': 'Jupyter Workflow Test Task', 'type': 'Task'}
